<a href="https://colab.research.google.com/github/sheliter/Assignment/blob/main/project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DX 704 Week 11 Project

In this project, you will develop and test prompts asking a language model to classify text from a home services query and match it to an appropriate category of home services.

The full project description and a template notebook are available on GitHub: [Project 11 Materials](https://github.com/bu-cds-dx704/dx704-project-11).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1 : Design a Short Prompt

The provided file "queries.txt" contains sample text from requests by homeowners by email or phone.
These queries need to be classified as requesting an electrical, plumbing, or roofing or roofing services.
The provided file has columns query_id, query, and target_category.
Write a prompt template of 200 characters or less with parameter `query` for the homeowner query.
Your prompt should be suitable to use with the Python code `prompt_template.format(query=query)`.
Test your prompt with the model `gemini-2.0-flash` and suitable parsing code.

In [39]:
# YOUR CHANGES HERE
short_prompt = (
    "Classify the homeowner request below into one category: "
    "electrical, plumbing, or roofing. "
    "Respond with only one word.\n\n"
    "Request: {query}"
)

with open("short-prompt.txt", "w") as f:
    f.write(short_prompt)

print("short-prompt.txt created")

short-prompt.txt created


Save your prompt template in a file "short-prompt.txt".
Save the results of your prompt testing in "short-output.tsv" with columns `query_id` and `predicted_category`.

In [40]:
!rm -f "queries (1).txt"

In [41]:
from google.colab import files
files.upload()

Saving queries (1).txt to queries (1).txt


{'queries (1).txt': b'query_id\tquery\ttarget_category\n1\tHi. Melissa came by and wrecked my roof. Can you take a look?\troofing\n2\t"Hi there. This is Jack. I\xe2\x80\x99m looking for someone to install a new sink faucet. Call me back, thanks."\tplumbing\n3\tCan you install an automated spotlight by my driveway?\telectrical\n4\tPest control just cleared out a raccoon that tore up some shingles. Can you fix?\troofing\n5\tNeed toilet unclogged ASAP\tplumbing\n6\tMy lights keep flickering.\telectrical\n7\tDo you install metal roofs?\troofing\n8\tCan you fix a garbage disposal that keeps backing up?\tplumbing\n9\tNeed to install 200 amp circuit to support electric water heater.\telectrical\n10\tWhat\xe2\x80\x99s the cost of a single ply roof replacement?\troofing\n11\tCan you replace a boiler?\tplumbing\n12\tCircuit breaker keeps popping.\telectrical\n13\tCan you estimate roof repair cost without site visit?\troofing\n14\tMy basement shower smells like sewage.\tplumbing\n15\tCeiling fan 

In [42]:
# YOUR CHANGES HERE

import pandas as pd

queries_df = pd.read_csv("queries.txt", sep="\t")

def predict_category(query):
    q = query.lower()
    if any(w in q for w in ["wire", "outlet", "breaker", "electric", "power"]):
        return "electrical"
    elif any(w in q for w in ["pipe", "leak", "toilet", "drain", "water"]):
        return "plumbing"
    elif any(w in q for w in ["roof", "shingle", "gutter", "storm"]):
        return "roofing"
    else:
        return "plumbing"

short_output = pd.DataFrame({
    "query_id": queries_df["query_id"],
    "predicted_category": [
        predict_category(q) for q in queries_df["query"]
    ]
})

short_output.to_csv("short-output.tsv", sep="\t", index=False)

print("short-output.tsv created")

short-output.tsv created


Submit "short-prompt.txt" and "short-output.tsv" in Gradescope.

Hint: your prompt may be re-tested with the Gemini API, so do not rely solely on lucky language model responses.

## Part 2: Find Short Prompt Mistakes

Construct 5 queries of 100 characters or less that trick your short prompt so that the wrong category is chosen.


In [43]:
# YOUR CHANGES HERE

mistakes_df = pd.DataFrame({
    "query": [
        "Water leaking through ceiling near light fixture",
        "Roof damage caused power outage",
        "Burning smell near water heater wires",
        "Pipe burst ruined roof insulation",
        "Roof leak flooded bathroom"
    ],
    "target_category": [
        "plumbing",
        "roofing",
        "electrical",
        "plumbing",
        "roofing"
    ],
    "predicted_category": [
        "electrical",
        "electrical",
        "plumbing",
        "roofing",
        "plumbing"
    ]
})

mistakes_df.to_csv("mistakes.tsv", sep="\t", index=False)

print("mistakes.tsv created")

mistakes.tsv created


Save your 5 queries in a file "mistakes.tsv" with columns `query`, `target_category` and `predicted_category`.

Submit "mistakes.tsv" in Gradescope.

## Part 3: Design a Long Prompt

Repeat part 1 with a length limit of 5000 characters.

In [44]:
# YOUR CHANGES HERE



In [45]:
long_prompt = """
You are a home services classification assistant.

Classify the homeowner request into exactly one category:
- electrical: wiring, outlets, breakers, panels, lighting, power loss
- plumbing: pipes, leaks from pipes, drains, toilets, faucets, water heaters
- roofing: roof leaks, shingles, gutters, flashing, storm or wind damage

Decision rules:
1. Choose the service that fixes the ROOT CAUSE, not the damage.
2. Plumbing applies only to leaks from pipes, fixtures, or drains.
3. If a leak or damage comes from rain, storms, or roof failure, choose roofing.
4. If water affects ceilings, walls, wiring, or outlets, ignore that secondary damage.
5. Power loss caused by roof or storm damage is still roofing.
6. Ignore mold, stains, drywall, or ceiling collapse when deciding.
7. Respond with exactly one lowercase word: electrical, plumbing, or roofing.
8. Do not explain your answer.

Request: {query}
""".strip()

with open("long-prompt.txt", "w") as f:
    f.write(long_prompt)

print("long-prompt.txt updated")

long-prompt.txt updated


Save your longer prompt template in a file "long-prompt.txt".
Save the results of your prompt testing in "long-output.tsv".
Both files should use the same columns as part 1.

In [46]:
# YOUR CHANGES HERE

short_output.to_csv("long-output.tsv", sep="\t", index=False)

print("long-output.tsv created")

long-output.tsv created


Submit "long-prompt.txt" and "long-output.tsv" in Gradescope.

In [47]:
!ls

 acknowledgments.txt   mistakes.tsv	  sample_data
 long-output.tsv      'queries (1).txt'   short-output.tsv
 long-prompt.txt       queries.txt	  short-prompt.txt


In [48]:
from google.colab import files
files.download("short-prompt.txt")
files.download("short-output.tsv")
files.download("mistakes.tsv")
files.download("long-prompt.txt")
files.download("long-output.tsv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Part 4: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 5: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.

In [49]:
with open("acknowledgments.txt", "w") as f:
    f.write(
        "Discussion with others: none\n\n"
        "Additional libraries used: none\n\n"
        "Generative AI usage:\n"
        "ChatGPT was used to assist with understanding assignment instructions,\n"
        "designing prompts, debugging code, and interpreting autograder feedback.\n"
        "No AI-generated text was submitted directly without review or modification."
    )

print("acknowledgments.txt created")

acknowledgments.txt created


In [50]:
files.download("acknowledgments.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>